In [1]:
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from multiprocessing import Pool

print("Libraries imported successfully!")
print("PyTorch version:", torch.__version__)

Libraries imported successfully!
PyTorch version: 2.13.0+cpu


In [2]:
# Winning combinations
LINES = [
    (0, 1, 2), (3, 4, 5), (6, 7, 8),
    (0, 3, 6), (1, 4, 7), (2, 5, 8),
    (0, 4, 8), (2, 4, 6)
]

# Find all empty positions
def moves(s):
    return [i for i in range(9) if s[i] == 0]

# Play a move and change perspective to the next player
def play(s, a):
    s = s.copy()
    s[a] = 1
    return -s

# Check whether the game has ended
def result(s):
    for a, b, c in LINES:
        if s[a] == s[b] == s[c] == -1:
            return -1.0

    if 0 not in s:
        return 0.0

    return None

print("Tic-Tac-Toe game functions created!")
print("Empty board:", np.zeros(9, dtype=int))
print("Available moves:", moves(np.zeros(9)))

Tic-Tac-Toe game functions created!
Empty board: [0 0 0 0 0 0 0 0 0]
Available moves: [0, 1, 2, 3, 4, 5, 6, 7, 8]


In [3]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()

        self.body = nn.Sequential(
            nn.Linear(9, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU()
        )

        self.pi = nn.Linear(64, 9)
        self.v = nn.Linear(64, 1)

    def forward(self, x):
        h = self.body(x)

        policy = self.pi(h)
        value = torch.tanh(self.v(h)).squeeze(-1)

        return policy, value


net = Net()

print(net)

Net(
  (body): Sequential(
    (0): Linear(in_features=9, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=64, bias=True)
    (3): ReLU()
  )
  (pi): Linear(in_features=64, out_features=9, bias=True)
  (v): Linear(in_features=64, out_features=1, bias=True)
)


In [4]:
def predict(net, s):
    with torch.no_grad():
        x = torch.tensor(s, dtype=torch.float32).unsqueeze(0)
        logits, value = net(x)

        probabilities = F.softmax(logits, dim=1)[0].numpy()

    # Remove probabilities for occupied cells
    mask = np.zeros(9)
    mask[moves(s)] = 1

    probabilities *= mask
    probabilities /= probabilities.sum()

    return probabilities, value.item()


test_board = np.zeros(9, dtype=np.float32)
policy, value = predict(net, test_board)

print("Policy probabilities:", np.round(policy, 3))
print("Value prediction:", round(value, 3))
print("Policy sum:", round(policy.sum(), 3))

Policy probabilities: [0.12  0.12  0.115 0.114 0.116 0.099 0.114 0.102 0.101]
Value prediction: -0.061
Policy sum: 1.0


In [5]:
class Node:
    def __init__(self, p):
        self.p = p
        self.n = 0
        self.w = 0.0
        self.kids = {}


def expand(net, node, s):
    p, v = predict(net, s)

    for a in moves(s):
        node.kids[a] = Node(p[a])

    return v


def mcts(net, s0, sims=50, c=1.5):
    root = Node(1.0)
    expand(net, root, s0)

    for _ in range(sims):
        node = root
        s = s0.copy()
        path = [root]

        # Selection
        while node.kids:
            parent_scale = math.sqrt(node.n + 1)

            def puct(item):
                action, child = item

                if child.n == 0:
                    q = 0.0
                else:
                    q = -child.w / child.n

                u = c * child.p * parent_scale / (1 + child.n)
                return q + u

            action, node = max(node.kids.items(), key=puct)

            s = play(s, action)
            path.append(node)

            if result(s) is not None:
                break

        # Evaluation or terminal result
        v = result(s)

        if v is None:
            v = expand(net, node, s)

        # Backpropagation: alternate the value's sign
        for current_node in reversed(path):
            current_node.n += 1
            current_node.w += v
            v = -v

    # Improved policy from visit counts
    pi = np.zeros(9)

    for action, child in root.kids.items():
        pi[action] = child.n

    return pi / pi.sum()


test_policy = mcts(net, np.zeros(9, dtype=np.float32), sims=50)

print("MCTS policy:", np.round(test_policy, 3))
print("Total probability:", round(test_policy.sum(), 3))
print("Most visited move:", int(np.argmax(test_policy)))

MCTS policy: [0.12 0.12 0.1  0.12 0.12 0.1  0.12 0.1  0.1 ]
Total probability: 1.0
Most visited move: 0


In [6]:
def self_play(net, sims=50):
    s = np.zeros(9, dtype=np.float32)
    history = []

    while True:
        pi = mcts(net, s, sims=sims)
        history.append((s.copy(), pi.copy()))

        # Sample a move from the MCTS policy
        action = np.random.choice(9, p=pi)
        s = play(s, action)

        r = result(s)

        if r is not None:
            break

    # If r == -1, the player who made the last move won.
    last_mover = (len(history) - 1) % 2

    examples = []

    for i, (state, policy) in enumerate(history):
        if r == 0:
            z = 0.0
        elif i % 2 == last_mover:
            z = 1.0
        else:
            z = -1.0

        examples.append((state, policy, z))

    return examples


game_data = self_play(net, sims=50)

print("Self-play game completed!")
print("Training examples generated:", len(game_data))
print("First board state:", game_data[0][0])
print("First policy:", np.round(game_data[0][1], 3))
print("First outcome:", game_data[0][2])

Self-play game completed!
Training examples generated: 6
First board state: [0. 0. 0. 0. 0. 0. 0. 0. 0.]
First policy: [0.12 0.12 0.1  0.12 0.12 0.1  0.12 0.1  0.1 ]
First outcome: -1.0


In [7]:
def train(net, data, steps=50):
    if not data:
        raise ValueError("Training data is empty.")

    optimizer = torch.optim.Adam(net.parameters(), lr=0.001)

    S = torch.tensor(
        np.array([d[0] for d in data]), dtype=torch.float32
    )
    P = torch.tensor(
        np.array([d[1] for d in data]), dtype=torch.float32
    )
    Z = torch.tensor(
        [d[2] for d in data], dtype=torch.float32
    )

    net.train()

    for _ in range(steps):
        logits, values = net(S)

        policy_loss = -(
            P * F.log_softmax(logits, dim=1)
        ).sum(dim=1).mean()

        value_loss = F.mse_loss(values, Z)

        loss = policy_loss + value_loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    net.eval()
    return loss.item()


loss = train(net, game_data, steps=50)

print("Training completed!")
print("Final training loss:", round(loss, 4))

Training completed!
Final training loss: 2.0191


In [8]:
if __name__ == "__main__":
    net = Net()
    buffer = []

    NUM_ITERATIONS = 10
    GAMES_PER_ITERATION = 32
    SIMULATIONS = 50
    TRAINING_STEPS = 50
    BUFFER_SIZE = 5000

    print("Starting AlphaZero Self-Play Training...\n")

    for iteration in range(NUM_ITERATIONS):
        # Generate self-play games
        for _ in range(GAMES_PER_ITERATION):
            game_examples = self_play(net, sims=SIMULATIONS)
            buffer.extend(game_examples)

        # Keep only recent examples
        buffer = buffer[-BUFFER_SIZE:]

        # Train the network
        loss = train(net, buffer, steps=TRAINING_STEPS)

        print(
            f"iter {iteration + 1}: "
            f"examples={len(buffer)}, "
            f"loss={loss:.3f}"
        )

    print("\nAlphaZero training completed!")

Starting AlphaZero Self-Play Training...

iter 1: examples=241, loss=2.500
iter 2: examples=454, loss=2.573
iter 3: examples=679, loss=2.408
iter 4: examples=906, loss=2.223
iter 5: examples=1140, loss=2.058
iter 6: examples=1367, loss=1.904
iter 7: examples=1620, loss=1.740
iter 8: examples=1875, loss=1.629
iter 9: examples=2129, loss=1.553
iter 10: examples=2379, loss=1.496

AlphaZero training completed!


In [9]:
test_board = np.zeros(9, dtype=np.float32)

policy, value = predict(net, test_board)

print("Trained Agent Evaluation")
print("------------------------")
print("Move probabilities:", np.round(policy, 3))
print("Predicted board value:", round(value, 3))

legal_moves = moves(test_board)
best_move = int(np.argmax(policy))

print("Legal moves:", legal_moves)
print("Recommended move:", best_move)
print("Is the recommended move legal?", best_move in legal_moves)

Trained Agent Evaluation
------------------------
Move probabilities: [0.079 0.074 0.027 0.016 0.572 0.116 0.092 0.008 0.015]
Predicted board value: 0.282
Legal moves: [0, 1, 2, 3, 4, 5, 6, 7, 8]
Recommended move: 4
Is the recommended move legal? True
